In [1]:
# -------------------------------
# 1. Imports
# -------------------------------
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Models
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# -------------------------------
# 2. Load Data
# -------------------------------
true = pd.read_csv('True.csv')
fake = pd.read_csv('Fake.csv')

true['label'] = 'REAL'
fake['label'] = 'FAKE'

df = pd.concat([true, fake]).sample(frac=1).reset_index(drop=True)

print("Dataset Loaded:", df.shape)

# -------------------------------
# 3. Split
# -------------------------------
X = df['text']
y = df['label']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)

# -------------------------------
# 4. TF-IDF Vectorization
# -------------------------------
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7, max_features=5000)

X_train = tfidf.fit_transform(x_train)
X_test = tfidf.transform(x_test)

# -------------------------------
# 5. Train Multiple Models
# -------------------------------
models = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=50),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB()
}

best_model = None
best_acc = 0
best_name = ""

print("\n🔍 Model Evaluation:\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    print(f"{name} Accuracy: {acc}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print("Classification Report:\n", classification_report(y_test, pred))
    print("-" * 50)

    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_name = name

print("\n🏆 Best Model:", best_name)
print("Best Accuracy:", best_acc)

# -------------------------------
# 6. Save Model
# -------------------------------
os.makedirs("model", exist_ok=True)

pickle.dump(best_model, open("model/model.pkl", "wb"))
pickle.dump(tfidf, open("model/vectorizer.pkl", "wb"))
pickle.dump(best_name, open("model/model_name.pkl", "wb"))

print("✅ Model saved successfully!")

Dataset Loaded: (44898, 5)

🔍 Model Evaluation:

Passive Aggressive Accuracy: 0.9946547884187082
Confusion Matrix:
 [[4630   28]
 [  20 4302]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       1.00      0.99      0.99      4658
        REAL       0.99      1.00      0.99      4322

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

--------------------------------------------------
Logistic Regression Accuracy: 0.9871937639198218
Confusion Matrix:
 [[4586   72]
 [  43 4279]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.99      0.98      0.99      4658
        REAL       0.98      0.99      0.99      4322

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

---------